# VisDiff Output Visualization

Display Group A (species present) and Group B (species absent) satellite images side-by-side with ranked habitat hypotheses from the VisDiff CSV output. Reconstructs image groups using the same scoring pipeline as `08_visdiff.py`.

## Section 1: Imports and Configuration

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path
from typing import List, Tuple

# Ensure the repo root is on the path so `mmocc` is importable regardless of kernel
_REPO_ROOT = Path(__file__).resolve().parents[2] if "__file__" in dir() else Path.cwd().parents[1]
# Fallback: resolve relative to this notebook's location
_NOTEBOOK_DIR = Path("/data/vision/beery/scratch/esierra/text-sat-hab-mmocc/sat_mmocc/analysis")
_REPO_ROOT = _NOTEBOOK_DIR.parents[1]
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from PIL import Image

from mmocc.config import cache_path, default_image_backbone, default_sat_backbone
from mmocc.interpretability_utils import (
    compute_site_scores,
    load_fit_results,
    rank_image_groups,
    resolve_fit_results_path,
)
from mmocc.utils import get_taxon_map

# ── Configuration ──────────────────────────────────────────────────────────────

# Path to the VisDiff descriptions CSV produced by 08_visdiff.py
VISDIFF_CSV = cache_path / "visdiff_naip_prompt1.csv"

# Satellite imagery source: "sentinel" or "naip"
IMAGERY_SOURCE = "naip"
PNG_DIRS = {
    "sentinel": cache_path / "sat_images_png",
    "naip": cache_path / "naip_images_png",
}
PNG_DIR = PNG_DIRS.get(IMAGERY_SOURCE, cache_path / f"{IMAGERY_SOURCE}_images_png")

# VisDiff / scoring parameters (must match those used in 08_visdiff.py)
MODALITIES = ["image", "sat", "covariates"]
IMAGE_BACKBONE = default_image_backbone
SAT_BACKBONE = default_sat_backbone
TOP_K = 50          # top-k sites used when building groups
UNIQUE_WEIGHT = 2.0 # weight used in "unique" mode

# Display parameters
N_IMAGES_DISPLAY = 6    # images per group shown in the grid
N_HYPOTHESES = 10       # top-N hypotheses to display per species
THUMB_SIZE = (128, 128) # thumbnail resize for quick display

taxon_map = get_taxon_map()
print(f"Repo root:     {_REPO_ROOT}")
print(f"PNG directory: {PNG_DIR}")
print(f"VisDiff CSV:   {VISDIFF_CSV}")
print(f"Taxon map entries: {len(taxon_map)}")

ModuleNotFoundError: No module named 'dotenv'

## Section 2: Load VisDiff CSV Output

In [ ]:
visdiff_df = pd.read_csv(VISDIFF_CSV)
visdiff_df["auroc"] = pd.to_numeric(visdiff_df["auroc"], errors="coerce")

print(f"Loaded {len(visdiff_df)} hypotheses for {visdiff_df['taxon_id'].nunique()} species")
print(f"Columns: {list(visdiff_df.columns)}")
display(visdiff_df.head(10))

In [ ]:
# ── Species overview ──────────────────────────────────────────────────────────
species_summary = (
    visdiff_df.groupby(["taxon_id", "species"])
    .agg(n_hypotheses=("difference", "count"), mean_auroc=("auroc", "mean"))
    .sort_values("mean_auroc", ascending=False)
    .reset_index()
)
print(f"\nSpecies with VisDiff results ({len(species_summary)} total):")
display(species_summary)

# ── AUROC distribution ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(visdiff_df["auroc"].dropna(), bins=30, color="steelblue", edgecolor="white")
ax.set_xlabel("AUROC")
ax.set_ylabel("Count")
ax.set_title("Distribution of VisDiff hypothesis AUROC scores")
plt.tight_layout()
plt.show()

## Section 3: Select Species and Reconstruct Image Groups

Calls the same scoring pipeline used by `08_visdiff.py` to recover which sites ended up in Group A and Group B for each species.

Set `FOCAL_TAXON_ID` to a single taxon ID to inspect one species, or set it to `None` to iterate all species in the CSV.

In [ ]:
# Set to a specific taxon_id string to view one species, or None for all species in CSV
FOCAL_TAXON_ID: str | None = None  # e.g. "12345"

available_taxon_ids = species_summary["taxon_id"].tolist()
focal_ids = [FOCAL_TAXON_ID] if FOCAL_TAXON_ID else available_taxon_ids
print(f"Will reconstruct image groups for {len(focal_ids)} species")

In [ ]:
def get_species_image_groups(
    taxon_id: str,
    modalities: List[str] = MODALITIES,
    image_backbone: str = IMAGE_BACKBONE,
    sat_backbone: str = SAT_BACKBONE,
    top_k: int = TOP_K,
    unique_weight: float = UNIQUE_WEIGHT,
    mode: str = "standard",
    png_dir: Path = PNG_DIR,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """Return (positives_df, negatives_df, display_name) for a single taxon.

    Mirrors the logic in run_species_visdiff_job so the groups match what VisDiff saw.
    """
    fit_path, resolved_modalities, resolved_image, resolved_sat = (
        resolve_fit_results_path(taxon_id, modalities, image_backbone, sat_backbone)
    )
    fit_results = load_fit_results(fit_path)
    site_scores, display_name = compute_site_scores(
        taxon_id, resolved_modalities, resolved_image, resolved_sat, fit_results
    )
    site_scores["image_path"] = site_scores["loc_id"].apply(
        lambda lid: str(png_dir / f"{lid}.png")
    )
    site_scores["image_exists"] = site_scores["image_path"].apply(
        lambda p: Path(p).exists()
    )
    positives_df, negatives_df = rank_image_groups(
        site_scores,
        resolved_modalities,
        mode=mode,
        unique_weight=unique_weight,
        top_k=top_k,
        image_modality="sat",
        test=False,
    )
    return positives_df, negatives_df, display_name


# Build a cache: taxon_id → {mode: (positives_df, negatives_df), display_name}
species_groups: dict = {}

for taxon_id in focal_ids:
    taxon_id_str = str(taxon_id)
    try:
        groups = {}
        display_name = None
        for mode in ("standard", "unique"):
            pos_df, neg_df, display_name = get_species_image_groups(
                taxon_id_str, mode=mode
            )
            groups[mode] = (pos_df, neg_df)
        species_groups[taxon_id_str] = {"groups": groups, "display_name": display_name}
        pos_count = len(groups["standard"][0])
        neg_count = len(groups["standard"][1])
        print(f"  {display_name} ({taxon_id_str}): {pos_count} positives, {neg_count} negatives (standard)")
    except FileNotFoundError as exc:
        warnings.warn(f"Skipping {taxon_id_str}: {exc}")

print(f"\nSuccessfully loaded groups for {len(species_groups)} species.")

## Section 4: Display Side-by-Side Image Grids per Species

Group A = sites where the species is likely **present** (high occupancy score).  
Group B = sites where the species is likely **absent** (low occupancy score).

In [ ]:
def load_thumb(path: str, size: Tuple[int, int] = THUMB_SIZE) -> np.ndarray | None:
    """Load an image as an RGB numpy array thumbnail, returning None on failure."""
    try:
        img = Image.open(path).convert("RGB").resize(size, Image.LANCZOS)
        return np.asarray(img)
    except Exception:
        return None


def plot_image_grid(
    image_paths: List[str],
    group_label: str,
    color: str,
    axes_row: list,
    n: int = N_IMAGES_DISPLAY,
) -> None:
    """Fill a row of axes with thumbnails from image_paths."""
    for col_idx, ax in enumerate(axes_row[:n]):
        if col_idx < len(image_paths):
            arr = load_thumb(image_paths[col_idx])
            if arr is not None:
                ax.imshow(arr)
                ax.set_title(
                    Path(image_paths[col_idx]).stem, fontsize=6, color="dimgray"
                )
            else:
                ax.text(0.5, 0.5, "missing", ha="center", va="center", fontsize=8)
        else:
            ax.set_visible(False)
        ax.axis("off")
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
            spine.set_visible(True)


def plot_species_image_grid(
    taxon_id: str,
    mode: str = "standard",
    n_images: int = N_IMAGES_DISPLAY,
) -> None:
    """Plot Group A / Group B image grid for a single species."""
    data = species_groups.get(str(taxon_id))
    if data is None:
        print(f"No data for taxon_id={taxon_id}")
        return
    pos_df, neg_df = data["groups"][mode]
    display_name = data["display_name"]
    pos_paths = pos_df["image_path"].tolist()
    neg_paths = neg_df["image_path"].tolist()

    n_cols = n_images
    fig, axes = plt.subplots(
        2, n_cols,
        figsize=(n_cols * 2.2, 5),
        gridspec_kw={"hspace": 0.08, "wspace": 0.05},
    )
    # Ensure axes is always 2D
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    fig.suptitle(
        f"{display_name}  (taxon {taxon_id}, mode={mode})",
        fontsize=13, fontweight="bold", y=1.01,
    )

    # Row labels
    for row_idx, (label, color, paths) in enumerate(
        [
            ("Group A — Present", "#2196F3", pos_paths),
            ("Group B — Absent", "#F44336", neg_paths),
        ]
    ):
        axes[row_idx, 0].set_ylabel(
            label, fontsize=10, color=color, fontweight="bold",
            rotation=90, labelpad=4,
        )
        plot_image_grid(paths, label, color, axes[row_idx].tolist(), n=n_cols)

    plt.tight_layout()
    plt.show()


# Preview the first species (standard mode)
if species_groups:
    first_id = next(iter(species_groups))
    plot_species_image_grid(first_id, mode="standard")

## Section 5: Display VisDiff Hypotheses per Species

Ranked habitat concepts from the VisDiff CSV, sorted by AUROC (higher = more discriminative).

In [ ]:
def plot_hypotheses(taxon_id: str, n: int = N_HYPOTHESES) -> None:
    """Horizontal bar chart of top-N VisDiff hypotheses for a species."""
    df = (
        visdiff_df[visdiff_df["taxon_id"].astype(str) == str(taxon_id)]
        .sort_values("auroc", ascending=False)
        .head(n)
    )
    if df.empty:
        print(f"No hypotheses found for taxon_id={taxon_id}")
        return

    display_name = df["species"].iloc[0]
    auroc_vals = df["auroc"].values[::-1]
    labels = df["difference"].values[::-1]

    # Truncate long labels for display
    max_label_len = 60
    labels_display = [
        (lbl if len(lbl) <= max_label_len else lbl[: max_label_len - 1] + "…")
        for lbl in labels
    ]

    cmap = plt.cm.RdYlGn
    colors = [cmap(v) for v in np.clip((auroc_vals - 0.4) / 0.4, 0, 1)]

    fig, ax = plt.subplots(figsize=(9, max(3, 0.42 * len(labels))))
    bars = ax.barh(range(len(labels_display)), auroc_vals, color=colors, edgecolor="white")
    ax.set_yticks(range(len(labels_display)))
    ax.set_yticklabels(labels_display, fontsize=9)
    ax.set_xlabel("AUROC")
    ax.axvline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_xlim(left=max(0, auroc_vals.min() - 0.05), right=min(1.0, auroc_vals.max() + 0.05))
    ax.set_title(
        f"Top-{n} VisDiff habitat hypotheses — {display_name} ({taxon_id})",
        fontsize=11, fontweight="bold",
    )
    # Annotate values
    for bar, val in zip(bars, auroc_vals):
        ax.text(
            bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8,
        )
    plt.tight_layout()
    plt.show()


# Preview the first species
if species_groups:
    plot_hypotheses(first_id)

## Section 6: Combined Species Report — Images + Ranked Hypotheses

Full per-species card: Group A/B image thumbnails on the left, top-N VisDiff hypotheses as a ranked bar chart on the right.

An **interactive dropdown** is provided via `ipywidgets` so you can flip through species without re-running cells. If ipywidgets is not available the notebook falls back to rendering all species statically.

In [ ]:
def plot_species_report(
    taxon_id: str,
    mode: str = "standard",
    n_images: int = N_IMAGES_DISPLAY,
    n_hyp: int = N_HYPOTHESES,
) -> None:
    """Render a combined figure: image grid (left) + hypothesis bar chart (right)."""
    data = species_groups.get(str(taxon_id))
    if data is None:
        print(f"No data for taxon_id={taxon_id}")
        return

    pos_df, neg_df = data["groups"][mode]
    display_name = data["display_name"]
    pos_paths = pos_df["image_path"].tolist()
    neg_paths = neg_df["image_path"].tolist()

    hyp_df = (
        visdiff_df[visdiff_df["taxon_id"].astype(str) == str(taxon_id)]
        .sort_values("auroc", ascending=False)
        .head(n_hyp)
    )
    auroc_vals = hyp_df["auroc"].values[::-1]
    labels = hyp_df["difference"].values[::-1]
    max_label_len = 55
    labels_display = [
        (lbl if len(lbl) <= max_label_len else lbl[: max_label_len - 1] + "…")
        for lbl in labels
    ]

    # ── Layout: left = image grids, right = hypothesis chart ──────────────────
    n_img_cols = n_images
    n_hyp_rows = len(labels_display)

    fig = plt.figure(figsize=(n_img_cols * 2.0 + 8, max(6, 0.45 * n_hyp_rows + 3)))
    fig.suptitle(
        f"{display_name}  (taxon {taxon_id}, mode={mode})",
        fontsize=13, fontweight="bold", y=1.00,
    )

    outer = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[n_img_cols * 2, 8], wspace=0.35)

    # Left: two rows (Group A on top, Group B on bottom)
    left_gs = gridspec.GridSpecFromSubplotSpec(2, n_img_cols, subplot_spec=outer[0], hspace=0.08, wspace=0.05)
    for row_idx, (label, color, paths) in enumerate(
        [("Group A — Present", "#2196F3", pos_paths), ("Group B — Absent", "#F44336", neg_paths)]
    ):
        axes_row = [fig.add_subplot(left_gs[row_idx, c]) for c in range(n_img_cols)]
        axes_row[0].set_ylabel(label, fontsize=9, color=color, fontweight="bold", rotation=90, labelpad=4)
        plot_image_grid(paths, label, color, axes_row, n=n_img_cols)

    # Right: hypothesis bar chart
    ax_hyp = fig.add_subplot(outer[1])
    if len(auroc_vals):
        cmap = plt.cm.RdYlGn
        colors = [cmap(v) for v in np.clip((auroc_vals - 0.4) / 0.4, 0, 1)]
        bars = ax_hyp.barh(range(n_hyp_rows), auroc_vals, color=colors, edgecolor="white")
        ax_hyp.set_yticks(range(n_hyp_rows))
        ax_hyp.set_yticklabels(labels_display, fontsize=8)
        ax_hyp.set_xlabel("AUROC")
        ax_hyp.axvline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
        ax_hyp.set_xlim(
            left=max(0, auroc_vals.min() - 0.05),
            right=min(1.0, auroc_vals.max() + 0.05),
        )
        ax_hyp.set_title("Top VisDiff hypotheses (Group A > Group B)", fontsize=9)
        for bar, val in zip(bars, auroc_vals):
            ax_hyp.text(
                bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=7.5,
            )
    else:
        ax_hyp.text(0.5, 0.5, "No hypotheses", ha="center", va="center")
        ax_hyp.axis("off")

    plt.tight_layout()
    plt.show()


# ── Render a preview for the first species ─────────────────────────────────────
if species_groups:
    plot_species_report(first_id)

In [ ]:
# ── Interactive dropdown (requires ipywidgets) ─────────────────────────────────
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display

    dropdown_options = [
        (f"{data['display_name']} ({tid})", tid)
        for tid, data in species_groups.items()
    ]
    mode_options = ["standard", "unique"]

    species_dropdown = widgets.Dropdown(
        options=dropdown_options,
        description="Species:",
        layout=widgets.Layout(width="450px"),
    )
    mode_dropdown = widgets.Dropdown(
        options=mode_options,
        value="standard",
        description="Mode:",
        layout=widgets.Layout(width="200px"),
    )
    out = widgets.Output()

    def _on_change(change):
        out.clear_output(wait=True)
        with out:
            plot_species_report(species_dropdown.value, mode=mode_dropdown.value)

    species_dropdown.observe(_on_change, names="value")
    mode_dropdown.observe(_on_change, names="value")

    controls = widgets.HBox([species_dropdown, mode_dropdown])
    ipy_display(widgets.VBox([controls, out]))

    # Trigger initial render
    with out:
        if species_groups:
            plot_species_report(next(iter(species_groups)), mode="standard")

except ImportError:
    print("ipywidgets not available — rendering all species statically instead.")
    for tid in list(species_groups)[:5]:  # limit to first 5 to avoid long output
        plot_species_report(tid)

### Optional: Render All Species (batch / static)

Uncomment and run the cell below to produce a report for every species in `species_groups`.

In [ ]:
# Uncomment to render all species
# for tid in species_groups:
#     plot_species_report(tid, mode="standard")